# PII Anonymization Techniques Demo

This notebook demonstrates various anonymization methods implemented based on FSD guidelines and academic research.

The techniques covered include:
1. **Removal techniques** - Variable removal and record suppression
2. **Pseudonymization** - Hash-based and systematic pseudonymization
3. **Recoding/Categorization** - Age, income, and date generalization
4. **Randomization** - Statistical noise and permutation techniques
5. **Text anonymization** - Pattern masking and redaction
6. **Statistical disclosure control** - K-anonymity enforcement
7. **Comprehensive workflow** - Complete anonymization pipeline

## Setup and Import Dependencies

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add the src directory to the path for imports
src_path = Path().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import after path setup to avoid import order issues
from pii_detector.core.anonymization import AnonymizationTechniques  # noqa: E402

## Create Sample Dataset

First, let's create a sample dataset containing various types of PII that we'll use for demonstration:

In [2]:
# Create sample data with various PII types
sample_data = pd.DataFrame(
    {
        "participant_id": ["P001", "P002", "P003", "P004", "P005"],
        "name": [
            "John Doe",
            "Jane Smith",
            "Alice Johnson",
            "Bob Wilson",
            "Carol Davis",
        ],
        "email": [
            "john@email.com",
            "jane@company.org",
            "alice@uni.edu",
            "bob@tech.com",
            "carol@health.net",
        ],
        "age": [25, 34, 29, 45, 38],
        "income": [45000, 75000, 52000, 95000, 68000],
        "city": ["Chicago", "New York", "Chicago", "Los Angeles", "Chicago"],
        "occupation": ["Engineer", "Teacher", "Engineer", "Manager", "Nurse"],
        "phone": ["555-1234", "555-5678", "555-9012", "555-3456", "555-7890"],
        "notes": [
            "Called about billing on March 3rd",
            "Prefers email contact",
            "Works at Chicago Tech Corp",
            "Manager at LA Consulting",
            "Nurse at Memorial Hospital",
        ],
    }
)

print("Original Data:")
display(sample_data)
print(f"\nDataset shape: {sample_data.shape}")

Original Data:


,participant_id,name,email,age,income,city,occupation,phone,notes
0,P001,John Doe,john@email.com,25,45000,Chicago,Engineer,555-1234,Called about billing on March 3rd
1,P002,Jane Smith,jane@company.org,34,75000,New York,Teacher,555-5678,Prefers email contact
2,P003,Alice Johnson,alice@uni.edu,29,52000,Chicago,Engineer,555-9012,Works at Chicago Tech Corp
3,P004,Bob Wilson,bob@tech.com,45,95000,Los Angeles,Manager,555-3456,Manager at LA Consulting
4,P005,Carol Davis,carol@health.net,38,68000,Chicago,Nurse,555-7890,Nurse at Memorial Hospital



Dataset shape: (5, 9)


## Initialize Anonymization Tools

In [3]:
# Initialize anonymization techniques with a fixed seed for reproducible results
anonymizer = AnonymizationTechniques(random_seed=42)
print("Anonymization tools initialized with random seed 42 for reproducible results.")

Anonymization tools initialized with random seed 42 for reproducible results.


## 1. Removal Techniques

The simplest anonymization approach is to completely remove identifying variables or records.

In [4]:
# Remove direct identifiers
step1 = anonymizer.remove_variables(sample_data, ["name", "email", "phone"])
print("After removing direct identifiers (name, email, phone):")
display(step1)

After removing direct identifiers (name, email, phone):


,participant_id,age,income,city,occupation,notes
0,P001,25,45000,Chicago,Engineer,Called about billing on March 3rd
1,P002,34,75000,New York,Teacher,Prefers email contact
2,P003,29,52000,Chicago,Engineer,Works at Chicago Tech Corp
3,P004,45,95000,Los Angeles,Manager,Manager at LA Consulting
4,P005,38,68000,Chicago,Nurse,Nurse at Memorial Hospital


In [5]:
# Remove records with unique combinations
unique_removed = anonymizer.remove_records_with_unique_combinations(
    sample_data, ["city", "occupation"], threshold=1
)
print(
    f"Records with unique city-occupation combinations removed: {len(sample_data) - len(unique_removed)}"
)
display(unique_removed)

Records with unique city-occupation combinations removed: 3


,participant_id,name,email,age,income,city,occupation,phone,notes
0,P001,John Doe,john@email.com,25,45000,Chicago,Engineer,555-1234,Called about billing on March 3rd
1,P003,Alice Johnson,alice@uni.edu,29,52000,Chicago,Engineer,555-9012,Works at Chicago Tech Corp


## 2. Pseudonymization Techniques

Replace identifying values with consistent pseudonyms that preserve relationships while removing direct identification.

In [6]:
step2 = step1.copy()

# Hash-based pseudonymization
step2["participant_id"] = anonymizer.hash_pseudonymization(
    step1["participant_id"], prefix="ANON_"
)
print("Hash-based pseudonymization of participant IDs:")
comparison_df = pd.DataFrame(
    {
        "original_id": step1["participant_id"],
        "pseudonymized_id": step2["participant_id"],
    }
)
display(comparison_df)

Hash-based pseudonymization of participant IDs:


,original_id,pseudonymized_id
0,P001,ANON_19e3112c
1,P002,ANON_59ed8689
2,P003,ANON_37902265
3,P004,ANON_ab3b01ed
4,P005,ANON_c852755e


## 3. Recoding/Categorization Techniques

Transform continuous variables into categories to reduce precision while preserving analytical utility.

In [7]:
step3 = step2.copy()

# Age categorization
step3["age_group"] = anonymizer.age_categorization(step2["age"])
print("Age categorization:")
age_comparison = pd.DataFrame(
    {"original_age": step2["age"], "age_group": step3["age_group"]}
)
display(age_comparison)

Age categorization:


,original_age,age_group
0,25,18-29
1,34,30-44
2,29,18-29
3,45,30-44
4,38,30-44


In [9]:
# Income categorization
step3["income_bracket"] = anonymizer.income_categorization(step2["income"])
print("Income categorization:")
income_comparison = pd.DataFrame(
    {"original_income": step2["income"], "income_bracket": step3["income_bracket"]}
)
display(income_comparison)

Income categorization:


,original_income,income_bracket
0,45000,Lower-Middle
1,75000,Middle
2,52000,Middle
3,95000,Upper-Middle
4,68000,Middle


In [10]:
# Top/bottom coding
step3["income_coded"] = anonymizer.top_bottom_coding(
    step2["income"], top_percentile=80, bottom_percentile=20
)
print("Top/bottom coding of income (80th/20th percentiles):")
coding_comparison = pd.DataFrame(
    {"original": step2["income"], "coded": step3["income_coded"]}
)
display(coding_comparison)

Top/bottom coding of income (80th/20th percentiles):


,original,coded
0,45000,≤50600
1,75000,75000
2,52000,52000
3,95000,≤50600
4,68000,68000


## 4. Randomization Techniques

Add controlled randomness to data while preserving statistical properties.

In [11]:
step4 = step3.copy()

# Add noise to numeric data
step4["age_with_noise"] = anonymizer.add_noise(
    step3["age"], noise_type="gaussian", noise_level=0.1
)
print("Gaussian noise added to age:")
noise_comparison = pd.DataFrame(
    {"original": step3["age"], "with_noise": step4["age_with_noise"].round(2)}
)
display(noise_comparison)

Gaussian noise added to age:


,original,with_noise
0,25,25.39
1,34,33.89
2,29,29.50
3,45,46.19
4,38,37.82


In [12]:
# Permutation swapping
swapped_data = anonymizer.permutation_swapping(
    step4, ["age", "income"], swap_probability=0.4
)
print("After permutation swapping (age and income):")
swap_comparison = pd.DataFrame(
    {
        "original_age": step4["age"],
        "swapped_age": swapped_data["age"],
        "original_income": step4["income"],
        "swapped_income": swapped_data["income"],
    }
)
display(swap_comparison)

After permutation swapping (age and income):


,original_age,swapped_age,original_income,swapped_income
0,25,38,45000,75000
1,34,29,75000,68000
2,29,34,52000,52000
3,45,45,95000,95000
4,38,25,68000,45000


## 5. Text Anonymization

Identify and mask PII patterns within text content.

In [13]:
# Text masking demonstration
sample_text = "John Doe called from 555-1234 about his account john@email.com"
masked_text = anonymizer.text_masking(sample_text)
print("Text masking example:")
print(f"Original: {sample_text}")
print(f"Masked:   {masked_text}")

Text masking example:
Original: John Doe called from 555-1234 about his account john@email.com
Masked:   [NAME] called from [PHONE] about his account [EMAIL]


In [14]:
# Apply to notes column
masked_notes = sample_data["notes"].apply(anonymizer.text_masking)
print("Original vs Masked notes:")
notes_comparison = pd.DataFrame(
    {"original_notes": sample_data["notes"], "masked_notes": masked_notes}
)
display(notes_comparison)

Original vs Masked notes:


,original_notes,masked_notes
0,Called about billing on March 3rd,Called about billing on March 3rd
1,Prefers email contact,Prefers email contact
2,Works at Chicago Tech Corp,Works at [NAME] Corp
3,Manager at LA Consulting,Manager at LA Consulting
4,Nurse at Memorial Hospital,Nurse at [NAME]


## 6. K-Anonymity Analysis

Ensure that each combination of quasi-identifiers appears for at least k individuals.

In [15]:
# Create test data for k-anonymity demonstration
test_data = pd.DataFrame(
    {
        "age_group": ["20-30", "20-30", "30-40", "30-40", "40-50"],
        "city": ["Chicago", "Chicago", "NYC", "NYC", "LA"],
        "occupation": ["Engineer", "Teacher", "Engineer", "Teacher", "Manager"],
        "salary": [50000, 45000, 75000, 65000, 95000],
    }
)

print("Test data for k-anonymity:")
display(test_data)

Test data for k-anonymity:


,age_group,city,occupation,salary
0,20-30,Chicago,Engineer,50000
1,20-30,Chicago,Teacher,45000
2,30-40,NYC,Engineer,75000
3,30-40,NYC,Teacher,65000
4,40-50,LA,Manager,95000


In [16]:
# Check k-anonymity
is_anonymous, violations = anonymizer.k_anonymity_check(
    test_data, ["age_group", "city"], k=2
)
print(f"Original data satisfies 2-anonymity: {is_anonymous}")
if not is_anonymous:
    print("\nViolations (combinations with < 2 records):")
    display(violations)

Original data satisfies 2-anonymity: False

Violations (combinations with < 2 records):


,age_group,city,count
2,40-50,LA,1


In [17]:
# Achieve k-anonymity
k_anonymous_data = anonymizer.achieve_k_anonymity(test_data, ["age_group", "city"], k=2)
is_now_anonymous, _ = anonymizer.k_anonymity_check(
    k_anonymous_data, ["age_group", "city"], k=2
)
print(f"After applying k-anonymity: {is_now_anonymous}")
print(f"Rows removed: {len(test_data) - len(k_anonymous_data)}")
print("\nFinal k-anonymous dataset:")
display(k_anonymous_data)

After applying k-anonymity: True
Rows removed: 1

Final k-anonymous dataset:


,age_group,city,occupation,salary
0,20-30,Chicago,Engineer,50000
1,20-30,Chicago,Teacher,45000
2,30-40,NYC,Engineer,75000
3,30-40,NYC,Teacher,65000


## 7. Comprehensive Anonymization Workflow

Apply multiple techniques in sequence for comprehensive anonymization.

In [18]:
# Apply full workflow
final_data = sample_data.copy()

print("Applying comprehensive anonymization workflow...")

# Step 1: Remove direct identifiers
final_data = anonymizer.remove_variables(final_data, ["name", "email", "phone"])
print("✓ Removed direct identifiers")

# Step 2: Pseudonymize IDs
final_data["participant_id"] = anonymizer.hash_pseudonymization(
    final_data["participant_id"], prefix="SUBJ_"
)
print("✓ Pseudonymized participant IDs")

# Step 3: Categorize continuous variables
final_data["age_group"] = anonymizer.age_categorization(final_data["age"])
final_data["income_bracket"] = anonymizer.income_categorization(final_data["income"])
final_data = final_data.drop(["age", "income"], axis=1)
print("✓ Categorized age and income, removed original values")

# Step 4: Anonymize text
final_data["notes"] = final_data["notes"].apply(anonymizer.text_masking)
print("✓ Anonymized text content")

# Step 5: Apply k-anonymity
final_data = anonymizer.achieve_k_anonymity(
    final_data, ["age_group", "city", "occupation"], k=2
)
print("✓ Enforced k-anonymity (k=2)")

print("\nFinal anonymized dataset:")
display(final_data)

Applying comprehensive anonymization workflow...
✓ Removed direct identifiers
✓ Pseudonymized participant IDs
✓ Categorized age and income, removed original values
✓ Anonymized text content
✓ Enforced k-anonymity (k=2)

Final anonymized dataset:


,participant_id,city,occupation,notes,age_group,income_bracket
0,SUBJ_19e3112c,Chicago,Engineer,Called about billing on March 3rd,18-29,Lower-Middle
1,SUBJ_37902265,Chicago,Engineer,Works at [NAME] Corp,18-29,Middle


## 8. Anonymization Report

Generate a comprehensive report comparing the original and anonymized datasets.

In [19]:
# Generate anonymization report
report = anonymizer.anonymization_report(sample_data, final_data)

print("=== ANONYMIZATION REPORT ===")
print(f"Original rows: {report['original_rows']}")
print(f"Anonymized rows: {report['anonymized_rows']}")
print(f"Rows removed: {report['rows_removed']} ({report['removal_percentage']:.1f}%)")

print("\nColumn transformations:")
for col, stats in report["columns_comparison"].items():
    if col in final_data.columns:
        print(
            f"  {col}: {stats['original_unique_values']} → {stats['anonymized_unique_values']} unique values "
            f"({stats['uniqueness_reduction']:.1f}% reduction)"
        )

=== ANONYMIZATION REPORT ===
Original rows: 5
Anonymized rows: 2
Rows removed: 3 (60.0%)

Column transformations:
  participant_id: 5 → 2 unique values (60.0% reduction)
  city: 3 → 1 unique values (66.7% reduction)
  occupation: 4 → 1 unique values (75.0% reduction)
  notes: 5 → 2 unique values (60.0% reduction)


## Summary

This notebook demonstrated comprehensive anonymization techniques including:

- **Variable removal** for direct identifiers
- **Hash-based pseudonymization** for consistent but anonymous IDs
- **Categorization** to reduce precision of continuous variables
- **Statistical noise** and **permutation** for randomization
- **Text masking** for PII within unstructured content
- **K-anonymity** enforcement for statistical disclosure control
- **Comprehensive reporting** for transparency and audit trails

These techniques can be combined and customized based on specific anonymization requirements and privacy regulations.